# Preparing state and national baby-name data

The study uses SSA baby-name counts at two levels: **state × recorded sex × year × name** and **recorded sex × year × name** nationally. The national documentation describes records as of 1 March 2026.

This notebook loads the original files, describes data quality, and adds variables for analysis. All published names and counts are retained. State and national tables remain separate because their geographic coverage and suppression differ.

Run the cells in order with the project environment. Prepared tables are saved in `data/processed/` for the EDA notebook.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Support starting from the repository, notebooks folder, or course folder.
cwd = Path.cwd().resolve()
ROOT = next(p for p in [cwd, cwd.parent, cwd / 'group_project_dsfm']
            if (p / 'data/raw/state').is_dir())
RAW = ROOT / 'data/raw/state'
NATIONAL_RAW = ROOT / 'data/raw/national'
OUT = ROOT / 'data/processed'
OUT.mkdir(exist_ok=True)

COLUMNS = ['state', 'sex', 'year', 'name', 'count']
KEY = ['state', 'sex', 'year', 'name']
GROUP = ['state', 'sex', 'year']
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 12)

## 1. Observation units and fields

| Field | Meaning |
|---|---|
| `state` | Two-letter state of birth, including DC; state data only |
| `sex` | Recorded sex (`F` or `M`) |
| `year` | Birth year |
| `name` | Published spelling; variants remain distinct |
| `count` | Published number of applications for the combination |

A state row is identified by `(state, sex, year, name)`; the national key omits state. Rows are aggregate counts, not individual people. Missing name records may be suppressed and are not treated as zero. See the [SSA qualifications](https://www.ssa.gov/oact/babynames/background.html) and [publication limits](https://www.ssa.gov/oact/babynames/limits.html).

## 2. Load the state files

The source files have no header. Column types are specified explicitly, and `keep_default_na=False` preserves literal name strings. Empty text fields are described separately below.

In [2]:
paths = sorted(RAW.glob('*.TXT'))
dtypes = {'state': 'string', 'sex': 'string', 'year': 'int32',
          'name': 'string', 'count': 'int64'}
data = pd.concat(
    [pd.read_csv(path, header=None, names=COLUMNS, dtype=dtypes,
                 keep_default_na=False) for path in paths],
    ignore_index=True
)
# Categories save memory for repeated labels without changing their values.
data = data.astype({'state': 'category', 'sex': 'category'})
print(f'{len(paths)} files | {len(data):,} records | '
      f'{data["state"].nunique()} states/DC | '
      f'{data["year"].min()}–{data["year"].max()}')
display(data.head(3))

51 files | 6,696,687 records | 51 states/DC | 1910–2025


,state,sex,year,name,count
0,AK,F,1910,Mary,14
1,AK,F,1910,Annie,12
2,AK,F,1910,Anna,10


## 3. Describe state data quality

Summarise missing or blank values, duplicate keys, recorded-sex codes and counts below the publication threshold of five. These summaries describe the data; no rows are automatically removed or corrected. Observed state–sex–year groups do not imply that every possible name is published.

In [3]:
state_quality = pd.Series({
    'Missing field values': data.isna().sum().sum(),
    'Blank text fields': sum(data[c].astype('string').str.strip().eq('').sum()
                             for c in ['state', 'sex', 'name']),
    'Excess duplicate-key rows': data.duplicated(KEY).sum(),
    'Invalid recorded-sex codes': (~data['sex'].isin(['F', 'M'])).sum(),
    'Counts below 5': data['count'].lt(5).sum(),
    'Years outside 1910–2025': (~data['year'].between(1910, 2025)).sum(),
}, name='Records or values')
display(state_quality.to_frame())

coverage = data.groupby(GROUP, observed=True).agg(
    records=('name', 'size'), published_count=('count', 'sum')
)
print(f'{data["year"].nunique()} observed years | '
      f'{len(coverage):,} state–sex–year groups | '
      f'{data["count"].sum():,} published counts')
display(coverage.head())

,Records or values
Missing field values,0
Blank text fields,0
Excess duplicate-key rows,0
Invalid recorded-sex codes,0
Counts below 5,0
Years outside 1910–2025,0


116 observed years | 11,832 state–sex–year groups | 330,978,761 published counts


records  published_count
state sex year                          
AK    F   1910        8               68
          1911        6               44
          1912       13               82
          1913        7               60
          1914       16              115

## 4. Review unusual names

Short or nonalphabetic spellings and placeholder-like labels are described without assuming they are errors. The named labels below are retained. Large counts can reflect popular names, so popularity alone is not a reason to remove a record.

In [4]:
names = data['name']
placeholder_labels = {'baby', 'babyboy', 'babygirl', 'unknown', 'unnamed', 'infant'}
placeholder = names.str.lower().isin(placeholder_labels)
state_name_review = pd.Series({
    'Surrounding whitespace': names.ne(names.str.strip()).sum(),
    'Fewer than two characters': names.str.len().lt(2).sum(),
    'Nonalphabetic spelling': (~names.str.fullmatch(r'[A-Za-z]+', na=False)).sum(),
    'Placeholder-like label': placeholder.sum(),
}, name='Records')
display(state_name_review.to_frame())
display(data.loc[placeholder].groupby('name').agg(
    records=('count', 'size'), published_count=('count', 'sum')
).sort_values('published_count', ascending=False))
display(data['count'].describe(percentiles=[0.5, 0.95, 0.99]).to_frame())
display(data.nlargest(5, 'count')[COLUMNS])

,Records
Surrounding whitespace,0
Fewer than two characters,0
Nonalphabetic spelling,0
Placeholder-like label,1811


,records,published_count
name,,
Unknown,785,15046
Baby,609,8955
Infant,134,7761
Babyboy,139,1323
Babygirl,123,1166
Unnamed,21,484


,count
count,6.696687e+06
mean,4.942425e+01
std,1.689818e+02
min,5.000000e+00
50%,1.200000e+01
95%,1.860000e+02
99%,6.530000e+02
max,1.002700e+04


,state,sex,year,name,count
4460953,NY,M,1947,Robert,10027
4460954,NY,M,1947,John,9639
4460248,NY,M,1946,Robert,9294
4475394,NY,M,1963,Michael,9236
4470264,NY,M,1958,Michael,9226


## 5. Add variables for analysis

All records and original spellings are retained.

| Variable | Definition |
|---|---|
| `decade` | Start year of the decade |
| `name_length` | Number of characters in the published spelling |
| `published_share` | Count divided by the published total within state × sex × year |
| `rank` | Count descending, with names ordered alphabetically for ties |

Shares refer to published counts, not all births. No additional count threshold or missing-name imputation is applied. The EDA notebook defines female shares separately.

In [5]:
prepared = data.sort_values(
    GROUP + ['count', 'name'], ascending=[True, True, True, False, True]
).copy()
prepared['decade'] = (prepared['year'] // 10 * 10).astype('int16')
prepared['name_length'] = prepared['name'].str.len().astype('int16')
groups = prepared.groupby(GROUP, observed=True, sort=False)
# Each denominator contains published counts from the same state, sex and year.
prepared['published_share'] = prepared['count'] / groups['count'].transform('sum')
prepared['rank'] = (groups.cumcount() + 1).astype('int32')
display(prepared.head(5))

,state,sex,year,name,count,decade,name_length,published_share,rank
0,AK,F,1910,Mary,14,1910,4,0.205882,1
1,AK,F,1910,Annie,12,1910,5,0.176471,2
2,AK,F,1910,Anna,10,1910,4,0.147059,3
3,AK,F,1910,Margaret,8,1910,8,0.117647,4
4,AK,F,1910,Helen,7,1910,5,0.102941,5


## 6. Save the state table

Save the nine-column table as a compressed CSV for the EDA notebook. The original files remain in `data/raw/state/`.

In [6]:
prepared.to_csv(
    OUT / 'state_names.csv.gz', index=False,
    compression={'method': 'gzip', 'compresslevel': 1}
)
print(f'Saved {len(prepared):,} records to data/processed/state_names.csv.gz')

Saved 6,696,687 records to data/processed/state_names.csv.gz


## 7. Load the national files

National files contain `name, sex, count`; the year comes from `yobYYYY.txt`. Searching the national folder recursively includes the `names/` subfolder. The table is kept separate from the state observations.

In [7]:
national_paths = sorted(NATIONAL_RAW.rglob('yob*.txt'))
NATIONAL_COLUMNS = ['year', 'sex', 'name', 'count']
NATIONAL_KEY = ['year', 'sex', 'name']
NATIONAL_GROUP = ['sex', 'year']
national_frames = []
for path in national_paths:
    frame = pd.read_csv(
        path, header=None, names=['name', 'sex', 'count'],
        dtype={'name': 'string', 'sex': 'string', 'count': 'int64'},
        keep_default_na=False
    )
    frame['year'] = int(path.stem[3:])
    national_frames.append(frame[NATIONAL_COLUMNS])
national = pd.concat(national_frames, ignore_index=True)
national = national.astype({'year': 'int32', 'sex': 'category'})
del national_frames, frame
print(f'{len(national_paths)} files | {len(national):,} records | '
      f'{national["year"].min()}–{national["year"].max()}')
display(national.head(3))

146 files | 2,181,032 records | 1880–2025


,year,sex,name,count
0,1880,F,Mary,7065
1,1880,F,Anna,2604
2,1880,F,Emma,2003


## 8. Describe national data quality

Use the same descriptive summaries as for state data. The national observation key is `(year, sex, name)`. Unusual names and placeholder-like records are retained, and absent name records are not filled with zeros.

In [8]:
national_quality = pd.Series({
    'Missing field values': national.isna().sum().sum(),
    'Blank text fields': sum(national[c].astype('string').str.strip().eq('').sum()
                             for c in ['sex', 'name']),
    'Excess duplicate-key rows': national.duplicated(NATIONAL_KEY).sum(),
    'Invalid recorded-sex codes': (~national['sex'].isin(['F', 'M'])).sum(),
    'Counts below 5': national['count'].lt(5).sum(),
    'Years outside 1880–2025': (~national['year'].between(1880, 2025)).sum(),
}, name='Records or values')
display(national_quality.to_frame())
national_coverage = national.groupby(NATIONAL_GROUP, observed=True).agg(
    records=('name', 'size'), published_count=('count', 'sum')
)
print(f'{national["year"].nunique()} observed years | '
      f'{len(national_coverage):,} sex–year groups | '
      f'{national["count"].sum():,} published counts')

national_names = national['name']
national_name_review = pd.Series({
    'Surrounding whitespace': national_names.ne(national_names.str.strip()).sum(),
    'Length outside README range 2–15': (~national_names.str.len().between(2, 15)).sum(),
    'Nonalphabetic spelling': (~national_names.str.fullmatch(r'[A-Za-z]+', na=False)).sum(),
    'Placeholder-like label': national_names.str.lower().isin(placeholder_labels).sum(),
}, name='Records')
display(national_name_review.to_frame())
display(national.nlargest(5, 'count')[NATIONAL_COLUMNS])

,Records or values
Missing field values,0
Blank text fields,0
Excess duplicate-key rows,0
Invalid recorded-sex codes,0
Counts below 5,0
Years outside 1880–2025,0


146 observed years | 292 sex–year groups | 375,362,447 published counts


,Records
Surrounding whitespace,0
Length outside README range 2–15,0
Nonalphabetic spelling,0
Placeholder-like label,655


,year,sex,name,count
431071,1947,F,Linda,99692
441441,1948,F,Linda,96220
437175,1947,M,James,94767
544684,1957,M,Michael,92781
437176,1947,M,Robert,91655


## 9. Prepare and save the national table

Add the same four derived fields. National shares use the published total within **sex × year**, and ranks order counts descending with alphabetical ties. The eight-column national table is saved separately from the state table.

In [9]:
national_prepared = national.sort_values(
    NATIONAL_GROUP + ['count', 'name'], ascending=[True, True, False, True]
).copy()
national_prepared['decade'] = (national_prepared['year'] // 10 * 10).astype('int16')
national_prepared['name_length'] = national_prepared['name'].str.len().astype('int16')
national_groups = national_prepared.groupby(NATIONAL_GROUP, observed=True, sort=False)
national_prepared['published_share'] = national_prepared['count'] / national_groups['count'].transform('sum')
national_prepared['rank'] = (national_groups.cumcount() + 1).astype('int32')

national_prepared.to_csv(
    OUT / 'national_names.csv.gz', index=False,
    compression={'method': 'gzip', 'compresslevel': 1}
)
print(f'Saved {len(national_prepared):,} records to data/processed/national_names.csv.gz')
display(national_prepared.head(5))

Saved 2,181,032 records to data/processed/national_names.csv.gz


,year,sex,name,count,decade,name_length,published_share,rank
0,1880,F,Mary,7065,1880,4,0.077642,1
1,1880,F,Anna,2604,1880,4,0.028617,2
2,1880,F,Emma,2003,1880,4,0.022012,3
3,1880,F,Elizabeth,1939,1880,9,0.021309,4
4,1880,F,Minnie,1746,1880,6,0.019188,5


## 10. Compare published totals for matching years

Join the national and pooled-state totals by recorded sex and year. The ratio describes these published datasets, not the proportion of all births covered. Geographic suppression, source scope and release differences can affect it. National and state counts are never added together; national years before 1910 remain available for national-only analyses.

In [11]:
pooled_state = coverage.groupby(['sex', 'year'], observed=True)['published_count'].sum()
comparison = pooled_state.rename('pooled_state_published_count').reset_index().merge(
    national_coverage['published_count'].rename('national_published_count').reset_index(),
    on=['sex', 'year'], how='inner'
).sort_values(['sex', 'year'])
comparison['national_minus_pooled_state'] = (
    comparison['national_published_count'] - comparison['pooled_state_published_count']
)
comparison['pooled_state_to_national_ratio'] = (
    comparison['pooled_state_published_count'] / comparison['national_published_count']
)

table_dir = ROOT / 'reports/tables'
table_dir.mkdir(exist_ok=True)
comparison.to_csv(table_dir / 'national_state_coverage.csv', index=False)
print(f'{len(comparison)} matching sex–year groups, '
      f'{comparison["year"].min()}–{comparison["year"].max()}')
display(comparison.loc[comparison['year'].isin([1910, 1950, 2025])])
display(comparison['pooled_state_to_national_ratio'].agg(['min', 'max']).to_frame(name='Ratio'))


232 matching sex–year groups, 1910–2025


,sex,year,pooled_state_published_count,national_published_count,national_minus_pooled_state,pooled_state_to_national_ratio
0,F,1910,352089,396503,44414,0.887986
40,F,1950,1610017,1713054,103037,0.939852
115,F,2025,1303470,1607267,303797,0.810985
116,M,1910,164222,194208,29986,0.845599
156,M,1950,1720112,1790262,70150,0.960816
231,M,2025,1466284,1708076,241792,0.858442


,Ratio
min,0.807590
max,0.963896
